# z605 - Tuning de hiperparametros (Optuna)
Mismos features (LGB05_FE608)

In [14]:
!pip install -q lightgbm pyarrow optuna

In [15]:
import os
import numpy as np
import polars as pl
import lightgbm as lgb
import optuna
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [16]:
PARAM = {
    'experimento': 'LGB05_FE608',
    'kaggle_competition': 'labo-iii-2026-ba',
    'base_path': '/home/ds/exp/FE608/',
    'archivo_features': 'tb_features_FE608.parquet',
    'apredecir_path': '/home/ds/datasets/product_id_apredecir201912.txt',
    'horizonte_meses': 2,
    'periodo_ultimo_dato': 201912,
    'periodo_target_final': 202002,
    'semilla': 102103,
    'n_trials': 50
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LGB05_FE608


## 1. Cargar features y armar target (identico al baseline 0.274)

In [17]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

df = pl.read_parquet(os.path.join(PARAM['base_path'], PARAM['archivo_features']))
df = df.sort(["product_id", "periodo"])

H = PARAM['horizonte_meses']

df = df.with_columns(
    pl.col("tn").shift(-H).over("product_id").alias("tn_target")
)
df = df.with_columns(
    (pl.col("periodo_m") + H).alias("periodo_target_m")
)

In [18]:
m_201910 = periodo_a_meses(201910)
m_201911 = periodo_a_meses(201911)
m_201912 = periodo_a_meses(201912)

df_valido = df.filter(pl.col("tn_target").is_not_null())

train = df_valido.filter(pl.col("periodo_target_m") <= m_201910)
valid = df_valido.filter(
    (pl.col("periodo_target_m") >= m_201911) & (pl.col("periodo_target_m") <= m_201912)
)

print("train:", train.height, " valid:", valid.height)

train: 27249  valid: 1827


In [19]:
cols_excluir = {"tn", "tn_target", "tn_shift1", "periodo", "periodo_target_m", "nacimiento_m"}
features = [c for c in df.columns if c not in cols_excluir]
categoricas = [c for c in ["product_id", "cat1", "cat2", "cat3", "brand", "descripcion"] if c in features]

def a_pandas(tabla):
    pdf = tabla.select(features + ["tn_target"]).to_pandas()
    for c in categoricas:
        pdf[c] = pdf[c].astype("category")
    return pdf

train_pd = a_pandas(train)
valid_pd = a_pandas(valid)

X_train = train_pd[features]
y_train = np.log1p(train_pd["tn_target"].clip(lower=0))

X_valid = valid_pd[features]
y_valid = np.log1p(valid_pd["tn_target"].clip(lower=0))

## 2. Busqueda Optuna
Cada trial entrena un LightGBM con early stopping sobre `valid` y devuelve el mejor rmse alcanzado. Optuna busca minimizar ese valor.

In [20]:
dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categoricas,
                      params={'feature_pre_filter': False})
dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=categoricas, reference=dtrain,
                      params={'feature_pre_filter': False})

def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'seed': PARAM['semilla'],
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'max_depth': trial.suggest_int('max_depth', -1, 15),
    }

    modelo = lgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    return modelo.best_score['valid_0']['rmse']

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']))
study.optimize(objective, n_trials=PARAM['n_trials'], show_progress_bar=True)

print("mejor rmse (log1p):", study.best_value)
print("mejores params:", study.best_params)

  0%|          | 0/50 [00:00<?, ?it/s]

mejor rmse (log1p): 0.49067626633014955
mejores params: {'learning_rate': 0.016498163113525355, 'num_leaves': 116, 'min_data_in_leaf': 102, 'feature_fraction': 0.6240128913238493, 'bagging_fraction': 0.5719394575069835, 'bagging_freq': 5, 'lambda_l1': 1.6393492816531436e-07, 'lambda_l2': 0.2566191777854699, 'max_depth': 15}


## 3. Reentrenar con los mejores hiperparametros

In [21]:
mejores_params = dict(study.best_params)
mejores_params.update({
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'seed': PARAM['semilla']
})

modelo_final = lgb.train(
    mejores_params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dtrain, dvalid],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=100)]
)

print("mejor iteracion:", modelo_final.best_iteration)

Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 0.525856	valid's rmse: 0.572094
[200]	train's rmse: 0.402267	valid's rmse: 0.491433
[300]	train's rmse: 0.373399	valid's rmse: 0.498179
Early stopping, best iteration is:
[217]	train's rmse: 0.395502	valid's rmse: 0.490676
mejor iteracion: 217


## 4. Prediccion para 202002 y submit

In [22]:
futuro = df.filter(pl.col("periodo") == PARAM['periodo_ultimo_dato'])
futuro_pd = futuro.select(features).to_pandas()
for c in categoricas:
    futuro_pd[c] = futuro_pd[c].astype("category")

pred_log = modelo_final.predict(futuro_pd, num_iteration=modelo_final.best_iteration)
pred_tn = np.expm1(pred_log)
pred_tn = np.clip(pred_tn, 0, None)

resultado = futuro.select(["product_id"]).to_pandas()
resultado["tn"] = pred_tn

In [23]:
apredecir = pl.read_csv(PARAM['apredecir_path'], separator="\t").to_pandas()

submit = apredecir[["product_id"]].merge(resultado, on="product_id", how="left")
print("nulos en submit (deberian ser 0):", submit["tn"].isna().sum())
submit["tn"] = submit["tn"].fillna(0.0)

archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)
submit.head()

nulos en submit (deberian ser 0): 0
/home/ds/exp/LGB05_FE608/LGB05_FE608_submit.csv


,product_id,tn
0,20001,949.625312
1,20002,936.860210
2,20003,793.703017
3,20004,581.967300
4,20005,558.710355


In [24]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} tuning Optuna sobre LGB05_FE608")

100%|██████████| 18.7k/18.7k [00:00<00:00, 57.5kB/s]


96 submissions remaining today.
Successfully submitted to Labo III, 2026 BA

In [33]:
importancia = pl.DataFrame({
    "feature": modelo_final.feature_name(),
    "importancia": modelo_final.feature_importance(importance_type="gain")
}).sort("importancia", descending=True)

importancia.write_csv(os.path.join(ruta, "feature_importance.csv"))
print(os.path.join(ruta, "feature_importance.csv"))

/home/ds/exp/LGB05_FE608/feature_importance.csv


In [26]:
print(df["ratio_cumplimiento"].max())
print(df["ratio_cumplimiento"].median())

sospechosos = df.filter((pl.col("cust_request_tn") == 0) & (pl.col("tn") > 0))
print("filas con demanda 0 pero venta > 0:", sospechosos.height)

1.365569231898702
0.9999996052814915
filas con demanda 0 pero venta > 0: 0


In [27]:
print(df.select([
    "cust_request_tn", "ratio_cumplimiento", "ratio_cumplimiento_lag1",
    "ratio_cumplimiento_media_3", "ratio_cumplimiento_media_12"
]).null_count())

shape: (1, 5)
┌─────────────────┬────────────────────┬───────────────────┬───────────────────┬───────────────────┐
│ cust_request_tn ┆ ratio_cumplimiento ┆ ratio_cumplimient ┆ ratio_cumplimient ┆ ratio_cumplimient │
│ ---             ┆ ---                ┆ o_lag1            ┆ o_media_3         ┆ o_media_12        │
│ u32             ┆ u32                ┆ ---               ┆ ---               ┆ ---               │
│                 ┆                    ┆ u32               ┆ u32               ┆ u32               │
╞═════════════════╪════════════════════╪═══════════════════╪═══════════════════╪═══════════════════╡
│ 0               ┆ 0                  ┆ 1233              ┆ 1233              ┆ 1233              │
└─────────────────┴────────────────────┴───────────────────┴───────────────────┴───────────────────┘


In [28]:
print("ratio_cumplimiento" in features)

True


In [31]:
nuevo = pd.read_csv("/home/ds/exp/LGB05_FE608/LGB05_FE608_submit.csv")
print(nuevo.columns.tolist())
print(nuevo.head())

['feature', 'importancia']
           feature    importancia
0  cust_request_tn  653160.108212
1      tn_media_12  216697.176631
2       tn_media_3   48638.651105
3       tn_media_9   39517.214037
4       tn_media_6   36204.734116


In [32]:
import os
print(os.listdir("/home/ds/exp/LGB05_FE608/"))

['LGB05_FE608_submit.csv']
